In [0]:
spark.catalog.clearCache()

In [0]:
spark.conf.set(
    "fs.azure.account.key.ecommercedeproj1998.dfs.core.windows.net",
    ""
)

In [0]:
fact_sales = spark.read.format("delta").load(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/fact_sales"
)
dim_date = spark.read.format("delta").load(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_date"
)
dim_product = spark.read.format("delta").load(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/dim_product"
)

In [0]:
from pyspark.sql.functions import sum as _sum

monthly_revenue = fact_sales.join(dim_date, "InvoiceDate") \
    .filter(fact_sales.TransactionType == "Sale") \
    .groupBy("Year", "Month") \
    .agg(_sum("TotalAmount").alias("TotalRevenue")) \
    .orderBy("Year", "Month")

monthly_revenue.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/agg_monthly_revenue"
)

monthly_revenue.show()

In [0]:
from pyspark.sql.functions import sum as _sum

monthly_revenue = fact_sales.join(dim_date, "InvoiceDate") \
    .filter(fact_sales.TransactionType == "Sale") \
    .groupBy("Year", "Month") \
    .agg(_sum("TotalAmount").alias("TotalRevenue")) \
    .orderBy("Year", "Month")

monthly_revenue.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/agg_monthly_revenue"
)
monthly_revenue.show()

In [0]:
top_products = fact_sales.join(dim_product, "StockCode") \
    .filter(fact_sales.TransactionType == "Sale") \
    .groupBy("Description") \
    .agg(_sum("TotalAmount").alias("TotalRevenue")) \
    .orderBy(_sum("TotalAmount").desc())

top_products.write.format("delta").mode("overwrite").save(
    "abfss://gold@ecommercedeproj1998.dfs.core.windows.net/agg_top_products"
)
top_products.show(10)